In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'triplet.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

# Calculating AVG

In [3]:
expansions = {
    'estimated_p': ['p0', 'p1'],
    's_': ['s0', 's1', 's2', 's3'],
    's__': ['s_0', 's_1', 's_2', 's_3'],
}

df = get_data_expanded(data.evaluation_data, expansions)
df_avgs = df.groupby('episode')[['p0', 'p1']].mean().reset_index()

def get_avg_per_epi(row):
    return df_avgs.loc[df_avgs['episode'] == row.episode][['p0', 'p1']].values[0]
df[['avg_p0', 'avg_p1']] = df.apply(get_avg_per_epi, axis=1, result_type='expand')

df[['episode', 'avg_p0', 'avg_p1']]

,episode,avg_p0,avg_p1
0,0,0.6000,-4.2675
1,0,0.6000,-4.2675
2,0,0.6000,-4.2675
3,0,0.6000,-4.2675
4,0,0.6000,-4.2675
...,...,...,...
2122,99,0.1448,0.8810
2123,99,0.1448,0.8810
2124,99,0.1448,0.8810
2125,99,0.1448,0.8810


In [4]:
df_std = df.groupby('episode')[['p0', 'p1']].std().reset_index()
df_min = df.groupby('episode')[['p0', 'p1']].min().reset_index()
df_max = df.groupby('episode')[['p0', 'p1']].max().reset_index()
df_count = df.groupby('episode')[['p0', 'p1']].count().reset_index()
df_stats = df_avgs[['episode']].copy()

df_stats[['avg_p0', 'avg_p1']] = df_avgs[['p0', 'p1']]
df_stats[['std_p0', 'std_p1']] = df_std[['p0', 'p1']]
df_stats[['min_p0', 'min_p1']] = df_min[['p0', 'p1']]
df_stats[['max_p0', 'max_p1']] = df_max[['p0', 'p1']]
df_stats['range_p0'] = df_stats['max_p0'] - df_stats['min_p0']
df_stats['range_p1'] = df_stats['max_p1'] - df_stats['min_p1']
df_stats[['count']] = df_count[['p0']]

df_stats

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
0,0,0.600000,-4.267500,0.477831,0.645375,-0.104,-5.244,1.070,-3.482,1.174,1.762,6
1,1,0.377000,0.428667,0.057236,0.079374,0.265,0.321,0.438,0.557,0.173,0.236,9
2,2,0.137937,0.484438,0.043571,0.063406,0.059,0.301,0.206,0.572,0.147,0.271,16
3,3,0.120895,0.987474,0.063311,0.084301,0.017,0.878,0.234,1.103,0.217,0.225,19
4,4,0.394846,-0.019462,0.080754,0.110116,0.238,-0.200,0.486,0.122,0.248,0.322,13
...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.607467,-2.633400,0.201035,0.415660,0.210,-3.236,0.863,-1.998,0.653,1.238,15
96,96,0.255395,0.998132,0.058771,0.036455,0.137,0.944,0.340,1.066,0.203,0.122,38
97,97,0.282343,0.254314,0.079103,0.137839,0.036,0.028,0.413,0.567,0.377,0.539,35
98,98,0.196916,0.628807,0.099796,0.102347,-0.037,0.422,0.359,0.881,0.396,0.459,83


In [5]:
df_stats.describe()

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000
mean,49.500000,0.400402,-0.468844,0.119934,0.185945,0.186100,-0.759140,0.560600,-0.161630,0.37450,0.597510,21.270000
std,29.011492,0.330129,2.103276,0.102572,0.248085,0.270818,2.341682,0.426827,1.846274,0.26410,0.731153,14.489882
min,0.000000,0.120895,-9.888333,0.022231,0.014806,-0.355000,-10.842000,0.190000,-9.704000,0.07100,0.050000,3.000000
25%,24.750000,0.199050,-1.009866,0.058387,0.055306,0.055500,-1.402500,0.320000,-0.620250,0.20450,0.190750,11.000000
50%,49.500000,0.303514,0.329419,0.079995,0.104685,0.137500,0.201000,0.418500,0.516000,0.26450,0.369000,16.500000
75%,74.250000,0.440300,0.817614,0.140194,0.219670,0.213250,0.719250,0.612000,0.907250,0.45975,0.708000,29.250000
max,99.000000,2.028667,1.744723,0.477831,1.941886,1.574000,1.624000,2.508000,2.029000,1.25600,5.335000,83.000000


# Infering with avg

In [6]:
import torch
import torch.nn as nn

m = model.transition_estimator.state_layer

input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'avg_p0', 'avg_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0288)

In [7]:
results = data.get_evaluation_metrics()
results.rse.mean()

np.float64(0.05157592853784673)

# Optimazing p

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

history = []

for epi, p1, p2 in df_avgs.values:
    print(f'{epi:}')
    d = df[df['episode'] == epi].copy().reset_index(drop=True)

    target_value = torch.tensor(d[['s_0', 's_1', 's_2', 's_3']].values)
    param = torch.tensor([p1, p2], requires_grad=True)
    print(f'initial value for input Param: {[round(p,4) for p in param.tolist()]}') 
    input_values = torch.tensor(d[['s0', 's1', 's2', 's3', 'a_']].values) 

    learning_rate = 0.1
    num_epochs = 500

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.repeat((input_values.shape[0], 1))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        # loss = criterion(output.float(), target_value.float())
        
        def normilize(v): 
            mins = input_values[:,:-1].min(axis=0).values.repeat((v.shape[0], 1))
            maxs = input_values[:,:-1].max(axis=0).values.repeat((v.shape[0], 1))
            rang = maxs - mins
            return (v - mins) / rang
        open_loss = criterion(normilize(output).float(), normilize(target_value).float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        # loss = torch.sqrt(open_loss.mean(axis=1).sum())
        # loss = open_loss.mean()
        # loss = open_loss.sum()

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # history.append((epoch, param.tolist(), loss.item()))

        # if (epoch + 1) % 100 == 0:
        #     # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
        #     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
    history.append({'episode': epi, 'optimized_p': param.tolist(), 'loss':  loss.item()})
    print(f'Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')


0.0
initial value for input Param: [0.6, -4.2675]
Loss: 0.3103, Input Param: [0.9273, -2.7906]
1.0
initial value for input Param: [0.377, 0.4287]
Loss: 0.0945, Input Param: [0.4019, 0.6938]
2.0
initial value for input Param: [0.1379, 0.4844]
Loss: 0.0723, Input Param: [0.2657, 0.6326]
3.0
initial value for input Param: [0.1209, 0.9875]
Loss: 0.0549, Input Param: [0.1049, 0.9033]
4.0
initial value for input Param: [0.3948, -0.0195]
Loss: 0.0937, Input Param: [0.2124, -0.5866]
5.0
initial value for input Param: [0.2796, 1.104]
Loss: 0.0413, Input Param: [0.3043, 1.3181]
6.0
initial value for input Param: [0.2757, 1.2687]
Loss: 0.0415, Input Param: [0.3027, 1.5962]
7.0
initial value for input Param: [0.55, -0.7478]
Loss: 0.1497, Input Param: [0.252, -1.7919]
8.0
initial value for input Param: [0.6442, -3.7216]
Loss: 0.7505, Input Param: [1.1314, -1.7163]
9.0
initial value for input Param: [0.2922, 0.4545]
Loss: 0.0416, Input Param: [0.2551, 0.4086]
10.0
initial value for input Param: [0.1

In [9]:
import pandas as pd
df_optim = pd.DataFrame(history)


expansions = {
    'optimized_p': ['opt_p0', 'opt_p1'],
}

df_optim = get_data_expanded(df_optim, expansions)[['episode', 'loss', 'opt_p0', 'opt_p1']]
df_optim.loss.mean()

np.float64(0.18088607622310518)

In [10]:
import torch
import torch.nn as nn

def get_opt_per_epi(row):
    return df_optim.loc[df_optim['episode'] == row.episode][['opt_p0', 'opt_p1']].values[0]
df[['opt_p0', 'opt_p1']] = df.apply(get_opt_per_epi, axis=1, result_type='expand')


input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'opt_p0', 'opt_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0301)

In [11]:
del model
del data